### https://www.kaggle.com/competitions/drawing-with-llms

In [58]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [59]:
import sys
sys.path.append('./utils')
sys.path.append('/home/vino/.cache/huggingface/hub')
from svg_processor import SVGSanitizer, SVGProcessor,svg_constraints
from siglip_class import SVGMetricEvaluator

In [60]:
### seed

In [61]:
import torch
import random
import numpy as np
import multiprocessing as mp

mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [62]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc


class Model:
    
    def __init__(self):

        self.model_path="meta-llama/Llama-3.2-1B-Instruct"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            gpu_memory_utilization=0.85,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )

       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded = self.get_response(descriptions)
        #base_svg_code = SVGProcessor.clean_and_extract_svgs(output_decoded, self.default_svg)
        #clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        #return SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)
        return output_decoded

In [63]:
model=Model()

WARNING 04-21 00:28:12 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-21 00:28:12 [config.py:585] This model supports multiple tasks: {'classify', 'score', 'reward', 'generate', 'embed'}. Defaulting to 'generate'.
INFO 04-21 00:28:12 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-21 00:28:13 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar', reasoning_b

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 04-21 00:28:16 [loader.py:447] Loading weights took 0.82 seconds
INFO 04-21 00:28:17 [gpu_model_runner.py:1186] Model loading took 2.3185 GB and 2.590044 seconds
INFO 04-21 00:28:21 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/d06a9ec75f/rank_0_0 for vLLM's torch.compile
INFO 04-21 00:28:21 [backends.py:425] Dynamo bytecode transform time: 3.95 s
INFO 04-21 00:28:21 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-21 00:28:23 [monitor.py:33] torch.compile takes 3.95 s in total
INFO 04-21 00:28:24 [kv_cache_utils.py:566] GPU KV cache size: 185,920 tokens
INFO 04-21 00:28:24 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 181.56x
INFO 04-21 00:28:36 [gpu_model_runner.py:1534] Graph capturing finished in 12 secs, took 0.29 GiB
INFO 04-21 00:28:36 [core.py:151] init engine (profile, create kv cache, warmup model) took 19.26 seconds


In [64]:
tmp=model.predict(['sun rising in the east'])

Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.97s/it, est. speed input: 12.49 


In [66]:
print(tmp)

['```svg\n                <svg width="400" height="400">\n                <circle cx="200" cy="200" r="100" fill="#f2f2f2" />\n                <circle cx="150" cy="150" r="50" fill="#f2f2f2" />\n                <circle cx="100" cy="100" r="30" fill="#f2f2f2" />\n                <circle cx="50" cy="50" r="10" fill="#f2f2f2" />\n                <circle cx="200" cy="200" r="100" fill="#f2f2f2" />\n                <circle cx="150" cy="150" r="50" fill="#f2f2f2" />\n                <circle cx="100" cy="100" r="30" fill="#f2f2f2" />\n                <circle cx="50" cy="50" r="10" fill="#f2f2f2" />\n                <rect x="0" y="0" width="100" height="200" rx="10" fill="#f2f2f2" />\n                <rect x="100" y="0" width="200" height="200" rx="10" fill="#f2f2f2" />\n                <line x1="100" y1="100" x2="200" y2="200" />\n                <line x1="150" y1="150" x2="200" y2="200" />\n                <line x1="200" y1="200" x2="150" y2="150" />\n                <line x1="200" y1="100" 

In [67]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test_vqa.csv',header=[0])
print(df.shape)
df.head(2)

(75, 7)


,description,gpt_svg,gpt_score_sl,response,vqa_pair,response_2,gpt_svg_2
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [ ]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [68]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 24
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


Batch prediction:   0%|                                   | 0/4 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/24 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   4%| | 1/24 [00:02<00:54,  2.36s/it, est. speed input: 25.82
cessed prompts:   8%| | 2/24 [00:02<00:23,  1.05s/it, est. speed input: 48.55
cessed prompts:  17%|▏| 4/24 [00:02<00:09,  2.03it/s, est. speed input: 87.05
cessed prompts:  21%|▏| 5/24 [00:03<00:07,  2.43it/s, est. speed input: 101.0
cessed prompts:  29%|▎| 7/24 [00:03<00:04,  3.82it/s, est. speed input: 133.0
cessed prompts:  33%|▎| 8/24 [00:03<00:06,  2.62it/s, est. speed input: 124.4
cessed prompts:  38%|▍| 9/24 [00:04<00:06,  2.34it/s, est. speed input: 122.9
cessed prompts:  46%|▍| 11/24 [00:05<00:04,  2.68it/s, est. speed input: 132.
cessed prompts:  50%|▌| 12/24 [00:05<00:03,  3.14it/s, est. speed input: 140.
cessed prompts:  54%|▌| 13/24 [00:05<00:04,  2.59it/s, est. speed input: 136.
Processed prompts: 100%|█| 24/24 [00:06<00:00,  3.70it/s, est

In [ ]:
model.close_model()

In [ ]:
#SigLip Score
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

In [ ]:
df['svg_score_3'].mean()

In [69]:
results

['```svg\n                <svg xmlns="http://www.w3.org/2000/svg">\n                  <defs>\n                    <linearGradient id="forest-gradient" x1="0" y1="0" x2="1" y2="1" gradientUnits="userSpaceOnUse">\n                      <stop offset="0" stopColor="#FFC107" stopOpacity="1"/>\n                      <stop offset="0.2" stopColor="#FF9800" stopOpacity="1"/>\n                      <stop offset="0.4" stopColor="#FFA07A" stopOpacity="1"/>\n                      <stop offset="0.6" stopColor="#FFA500" stopOpacity="1"/>\n                      <stop offset="0.8" stopColor="#FFC107" stopOpacity="1"/>\n                      <stop offset="0.9" stopColor="#FF9800" stopOpacity="1"/>\n                    </linearGradient>\n                  </defs>\n                  <g transform="translate(100, 100)">\n                    <rect x="0" y="0" width="100" height="100" fill="url(#forest-gradient)" rx="10"/>\n                    <g>\n                      <circle cx="20" cy="20" r="10" fill="#F